# iris_utils 模块测试
引用 [iris_utils.py](iris_utils.py) 并验证 `load_iris_ready()` 的功能：

1. 能正常导入并返回 4 组数据 + scaler
2. 数据形状正确（训练集 119 条，测试集 30 条）
3. 数据清洗生效（iris 原始 150 条 → 清洗后 149 条）
4. 标准化生效（训练集特征均值 ≈ 0，标准差 ≈ 1）
5. 随机种子生效（相同参数两次调用结果完全一致）

In [8]:
from my_datasets.iris_utils import load_iris_ready

In [9]:
# 测试 1+2+3：调用函数，验证形状与清洗效果
x_train, x_test, y_train, y_test, scaler = load_iris_ready()

print("训练集特征:", x_train.shape)   # 预期 (119, 4)：150 - 1条重复 = 149，149 × 0.8 ≈ 119
print("测试集特征:", x_test.shape)    # 预期 (30, 4)：149 × 0.2 ≈ 30
print("训练集标签:", y_train.shape, "| 唯一值:", y_train.unique().tolist())
print("\n训练集品种分布（应大致均衡）:")
print(y_train.value_counts())

assert x_train.shape == (119, 4), "训练集形状不对"
assert x_test.shape == (30, 4), "测试集形状不对"
assert set(y_train.unique()) == {'setosa', 'versicolor', 'virginica'}, "标签品种不全"
print("\n✓ 形状与标签验证通过")

训练集特征: (119, 4)
测试集特征: (30, 4)
训练集标签: (119,) | 唯一值: ['setosa', 'versicolor', 'virginica']

训练集品种分布（应大致均衡）:
species
setosa        44
versicolor    41
virginica     34
Name: count, dtype: int64

✓ 形状与标签验证通过


In [3]:
# 测试 4：标准化是否生效
print("训练集各特征均值（应≈0）:")
print(x_train.mean().round(6))
print("\n训练集各特征标准差（应≈1）:")
print(x_train.std().round(6))

assert (x_train.mean().abs() < 1e-6).all(), "均值不为0，标准化未生效"
print("\n✓ 标准化验证通过")

# scaler 是在训练集上 fit 好的转换器，可复用于新数据
print("\nscaler 对象:", scaler)

训练集各特征均值（应≈0）:
sepal length (cm)    0.0
sepal width (cm)    -0.0
petal length (cm)   -0.0
petal width (cm)     0.0
dtype: float64

训练集各特征标准差（应≈1）:
sepal length (cm)    1.004228
sepal width (cm)     1.004228
petal length (cm)    1.004228
petal width (cm)     1.004228
dtype: float64

✓ 标准化验证通过

scaler 对象: StandardScaler()


In [10]:
# 测试 5：random_state 固定后，两次调用结果应完全一致（可复现性）
a = load_iris_ready()
b = load_iris_ready()

import pandas as pd
assert a[0].equals(b[0]), "两次调用结果不一致！"
assert a[1].equals(b[1]), "两次测试集不一致！"
print("✓ 可复现性验证通过：相同 random_state 两次调用结果完全一致")

# 对比：scale=False 时返回未标准化的原始厘米数据，scaler 为 None
xtr_raw, *_ , sc_raw = load_iris_ready(scale=False)
print("\nscale=False 时单位为 cm 的训练集前2行:")
print(xtr_raw.head(2))
print("scaler =", sc_raw)

✓ 可复现性验证通过：相同 random_state 两次调用结果完全一致

scale=False 时单位为 cm 的训练集前2行:
    sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)
30                4.8               3.1                1.6               0.2
31                5.4               3.4                1.5               0.4
scaler = None
